# Step 2b: Train COOLANT with AdaBelief (Phased)

Phased pre-training with image-caption pairs using AdaBelief:
- Phase 1: CLIP pre-training (self-supervised, 10 epochs)
- Phase 2: Similarity pre-training (matched/unmatched, 10 epochs)
- Phase 3: Detection training (balanced pairs, 20 epochs)
- Phase 4: Joint fine-tuning (0.1x LR, 10 epochs)

AdaBelief config: `eps=1e-16, weight_decouple=True, rectify=True`

- Input: `./processed/coolant_{train,dev,test}.h5`
- Output: `./checkpoints_phased/`

In [ ]:
import subprocess, sys
try:
    from adabelief_pytorch import AdaBelief
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "adabelief-pytorch"])
    from adabelief_pytorch import AdaBelief

import os, math, random, warnings, csv, datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from sklearn.metrics import (
    f1_score, precision_recall_fscore_support,
    confusion_matrix, accuracy_score, classification_report,
)

warnings.filterwarnings("ignore")

# ── Colab / local path detection ──────────────────────────────────────────────
try:
    from google.colab import drive
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    drive.mount("/content/drive")
    project_root = Path("/content/drive/MyDrive/Thesis_Final/fake-new-detection")
else:
    # Local: this notebook lives at <project_root>/notebooks/workflow_coolant_adabelief/
    project_root = Path(__file__).resolve().parents[2] if "__file__" in dir() else Path().absolute().parents[1]

sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

print(f"ON_COLAB : {ON_COLAB}")
print(f"Project root: {project_root}")

In [ ]:
# Device, seeds, paths
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Absolute output dirs (relative to notebook folder inside project)
_nb_dir = project_root / "notebooks" / "workflow_coolant_adabelief"
SAVE_DIR = _nb_dir / "checkpoints_phased"
LOG_DIR  = _nb_dir / "logs"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device  : {DEVICE}")
print(f"SAVE_DIR: {SAVE_DIR}")
print(f"LOG_DIR : {LOG_DIR}")

In [ ]:
# Load data and create model — auto-discover HDF5 files and detect dimensions
import h5py
import glob
from src.processing.coolant import create_coolant_dataloaders
from src.processing.coolant.training_utils import make_coolant_pairs, make_detection_batch, soft_cross_entropy
from src.models.resnet_coolant import PatchedCOOLANT, patch_encoding, patch_clip_projection, patch_cnn_with_dropout

# Absolute path to the processed HDF5 directory
HDF5_DIR = project_root / "notebooks" / "workflow_coolant_adabelief" / "processed"
BATCH_SIZE = 32

print(f"HDF5_DIR: {HDF5_DIR}")

# Find the HDF5 files: coolant_<image_model>_<text_model>_train.h5
train_candidates = sorted(glob.glob(str(HDF5_DIR / "coolant_*_train.h5")))
if not train_candidates:
    raise FileNotFoundError(f"No coolant_*_train.h5 found in {HDF5_DIR}. Run 1_preprocess_from_pairs.ipynb first.")
if len(train_candidates) > 1:
    print(f"Found multiple feature sets:")
    for c in train_candidates:
        print(f"  {Path(c).name}")
    print(f"Using: {Path(train_candidates[-1]).name}  (latest)")

_train_h5 = train_candidates[-1]
_prefix = Path(_train_h5).name.replace("_train.h5", "")
_dev_h5 = str(HDF5_DIR / f"{_prefix}_dev.h5")
_test_h5 = str(HDF5_DIR / f"{_prefix}_test.h5")

with h5py.File(_train_h5, "r") as f:
    IMAGE_DIM = f["image_features"].shape[1]
    TEXT_EMBED = f["caption_features"].shape[2]
    _image_model = f.attrs.get("image_model", "unknown")
    _text_model = f.attrs.get("text_model", "unknown")

print(f"Features: {_prefix}")
print(f"IMAGE_DIM={IMAGE_DIM} ({_image_model}), TEXT_EMBED={TEXT_EMBED} ({_text_model})")

loaders, datasets = create_coolant_dataloaders(
    _train_h5, _dev_h5, _test_h5, batch_size=BATCH_SIZE,
)

CONFIG = {
    # Architecture
    "shared_dim": 128, "sim_dim": 64, "clip_embed_dim": 64,
    "feature_dim": 96, "h_dim": 64,
    # AdaBelief
    "lr": 3e-4,        # 3e-4 (matches stable AdamW baseline; 1e-3 causes mode collapse)
    "eps": 1e-16, "betas": (0.9, 0.999),
    "weight_decouple": True, "rectify": True, "weight_decay": 1e-4,
    # Training
    "label_smoothing": 0.1, "grad_clip": 1.0, "dropout": 0.3,
    # Phase durations
    "max_epochs_phase1": 10, "max_epochs_phase2": 10,
    "max_epochs_phase3": 20, "max_epochs_phase4": 10,
    "patience": 5,
    "seed": SEED, "batch_size": BATCH_SIZE,
}

model = PatchedCOOLANT(CONFIG)
patch_encoding(model.similarity_module.encoding, image_dim=IMAGE_DIM)
patch_encoding(model.detection_module.encoding, image_dim=IMAGE_DIM)
patch_encoding(model.detection_module.ambiguity_module.encoding, image_dim=IMAGE_DIM)
patch_clip_projection(model.clip_module, target_dim=IMAGE_DIM, is_image=True)
patch_clip_projection(model.clip_module, target_dim=TEXT_EMBED, is_image=False)
patch_cnn_with_dropout(model.similarity_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"])
patch_cnn_with_dropout(model.detection_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"])
patch_cnn_with_dropout(model.detection_module.ambiguity_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"])
model = model.to(DEVICE)

loss_cos = nn.CosineEmbeddingLoss(margin=0.2)
loss_ce = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
loss_ce_clip = nn.CrossEntropyLoss()  # CLIP: no smoothing
loss_kl = nn.KLDivLoss(reduction="batchmean")

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {n_params:,} params")
print(f"Train: {len(loaders['train'])} batches")
print(f"LR: {CONFIG['lr']} (Phase 4 joint: {CONFIG['lr'] * 0.1})")

In [ ]:
# Helper: create AdaBelief for a parameter set
def make_adabelief(params, lr=None):
    return AdaBelief(
        params,
        lr=lr or CONFIG["lr"],
        eps=CONFIG["eps"],
        betas=CONFIG["betas"],
        weight_decouple=CONFIG["weight_decouple"],
        rectify=CONFIG["rectify"],
        weight_decay=CONFIG["weight_decay"],
        print_change_log=False,
    )

# CSV logger
csv_path = LOG_DIR / "train_phased.csv"
csv_fields = ["phase", "epoch", "train_loss", "train_f1", "val_f1", "val_real_rec", "val_fake_rec"]
csv_file = open(csv_path, "w", newline="")
csv_writer = csv.DictWriter(csv_file, fieldnames=csv_fields)
csv_writer.writeheader()

print(f"Logging to {csv_path}")

In [ ]:
# ===== PHASE 1: CLIP Pre-training =====
print("=" * 50)
print("PHASE 1: CLIP Pre-training (AdaBelief)")
print("=" * 50)

for name, param in model.named_parameters():
    param.requires_grad = "clip_module" in name

optim_clip = make_adabelief(model.clip_module.parameters())
best_clip_loss = float("inf")

for epoch in range(1, CONFIG["max_epochs_phase1"] + 1):
    model.train()
    epoch_loss, n = 0.0, 0
    for caption, image, _ in tqdm(loaders["train"], desc=f"P1 Ep{epoch}"):
        caption, image = caption.to(DEVICE), image.to(DEVICE)
        bs = caption.size(0)
        if bs < 4: continue
        cap_pooled = caption.mean(dim=2)
        ie, te = model.clip_module(image, cap_pooled)
        logits = ie @ te.T * math.exp(0.07)
        ids = torch.arange(bs, device=DEVICE)
        lc = (loss_ce_clip(logits, ids) + loss_ce_clip(logits.T, ids)) / 2
        optim_clip.zero_grad(); lc.backward()
        torch.nn.utils.clip_grad_norm_(model.clip_module.parameters(), CONFIG["grad_clip"])
        optim_clip.step()
        epoch_loss += lc.item() * bs; n += bs

    avg_loss = epoch_loss / n
    print(f"  Epoch {epoch}: loss={avg_loss:.4f}")
    csv_writer.writerow({"phase": 1, "epoch": epoch, "train_loss": f"{avg_loss:.4f}",
                         "train_f1": "", "val_f1": "", "val_real_rec": "", "val_fake_rec": ""})
    csv_file.flush()

    if avg_loss < best_clip_loss:
        best_clip_loss = avg_loss
        torch.save(model.clip_module.state_dict(), SAVE_DIR / "clip_phase1.pth")

print(f"Phase 1 done. Best loss: {best_clip_loss:.4f}")

In [ ]:
# ===== PHASE 2: Similarity Pre-training =====
print("\n" + "=" * 50)
print("PHASE 2: Similarity Pre-training (AdaBelief)")
print("=" * 50)

for param in model.clip_module.parameters(): param.requires_grad = False
for param in model.similarity_module.parameters(): param.requires_grad = True
for param in model.detection_module.parameters(): param.requires_grad = False

optim_sim = make_adabelief(model.similarity_module.parameters())
best_sim_loss = float("inf")

for epoch in range(1, CONFIG["max_epochs_phase2"] + 1):
    model.train()
    epoch_loss, n_batches = 0.0, 0
    for caption, image, _ in tqdm(loaders["train"], desc=f"P2 Ep{epoch}"):
        caption, image = caption.to(DEVICE), image.to(DEVICE)
        if caption.size(0) < 4: continue
        cap_a, img_m, img_u = make_coolant_pairs(caption, image)
        ta_m, ia_m, _ = model.similarity_module(cap_a, img_m)
        ta_u, ia_u, _ = model.similarity_module(cap_a, img_u)
        t_cat = torch.cat([ta_m, ta_u])
        i_cat = torch.cat([ia_m, ia_u])
        y_cos = torch.cat([torch.ones(ta_m.size(0), device=DEVICE), -torch.ones(ta_u.size(0), device=DEVICE)])
        ls = loss_cos(t_cat, i_cat, y_cos)
        optim_sim.zero_grad(); ls.backward()
        torch.nn.utils.clip_grad_norm_(model.similarity_module.parameters(), CONFIG["grad_clip"])
        optim_sim.step()
        epoch_loss += ls.item(); n_batches += 1

    avg = epoch_loss / n_batches
    print(f"  Epoch {epoch}: loss={avg:.4f}")
    csv_writer.writerow({"phase": 2, "epoch": epoch, "train_loss": f"{avg:.4f}",
                         "train_f1": "", "val_f1": "", "val_real_rec": "", "val_fake_rec": ""})
    csv_file.flush()

    if avg < best_sim_loss:
        best_sim_loss = avg
        torch.save(model.similarity_module.state_dict(), SAVE_DIR / "sim_phase2.pth")

print(f"Phase 2 done. Best loss: {best_sim_loss:.4f}")

In [ ]:
# ===== PHASE 3: Detection Training =====
print("\n" + "=" * 50)
print("PHASE 3: Detection Training (AdaBelief)")
print("=" * 50)

for param in model.clip_module.parameters(): param.requires_grad = False
for param in model.similarity_module.parameters(): param.requires_grad = False
for param in model.detection_module.parameters(): param.requires_grad = True

optim_det = make_adabelief(model.detection_module.parameters())
best_val_f1 = 0.0
patience = 0

for epoch in range(1, CONFIG["max_epochs_phase3"] + 1):
    model.train()
    all_y, all_p = [], []
    for caption, image, _ in tqdm(loaders["train"], desc=f"P3 Ep{epoch}"):
        caption, image = caption.to(DEVICE), image.to(DEVICE)
        if caption.size(0) < 4: continue
        det_cap, det_img, det_labels = make_detection_batch(caption, image)
        cap_pooled = det_cap.mean(dim=2)
        with torch.no_grad():
            ie, te = model.clip_module(det_img, cap_pooled)
        det_out, attn, skl = model.detection_module(det_cap, det_img, te, ie)
        ld = loss_ce(det_out, det_labels) + 0.5 * loss_kl(F.log_softmax(attn, 1), F.softmax(skl, 1))
        optim_det.zero_grad(); ld.backward()
        torch.nn.utils.clip_grad_norm_(model.detection_module.parameters(), CONFIG["grad_clip"])
        optim_det.step()
        all_y.extend(det_labels.cpu().numpy()); all_p.extend(det_out.argmax(1).cpu().numpy())

    train_f1 = f1_score(all_y, all_p, average="macro", zero_division=0)

    # Validate
    model.eval()
    val_y, val_p = [], []
    with torch.no_grad():
        for caption, image, _ in loaders["dev"]:
            caption, image = caption.to(DEVICE), image.to(DEVICE)
            if caption.size(0) < 4: continue
            det_cap, det_img, det_labels = make_detection_batch(caption, image)
            cap_pooled = det_cap.mean(dim=2)
            ie, te = model.clip_module(det_img, cap_pooled)
            det_out, _, _ = model.detection_module(det_cap, det_img, te, ie)
            val_y.extend(det_labels.cpu().numpy()); val_p.extend(det_out.argmax(1).cpu().numpy())

    val_f1 = f1_score(val_y, val_p, average="macro", zero_division=0)
    prec, rec, _, _ = precision_recall_fscore_support(val_y, val_p, average=None, zero_division=0)
    cm = confusion_matrix(val_y, val_p)
    print(f"  Epoch {epoch}: train_F1={train_f1:.4f} val_F1={val_f1:.4f} CM={cm.tolist()}")

    csv_writer.writerow({"phase": 3, "epoch": epoch, "train_loss": "",
                         "train_f1": f"{train_f1:.4f}", "val_f1": f"{val_f1:.4f}",
                         "val_real_rec": f"{rec[0]:.3f}" if len(rec) > 0 else "",
                         "val_fake_rec": f"{rec[1]:.3f}" if len(rec) > 1 else ""})
    csv_file.flush()

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1; patience = 0
        torch.save(model.detection_module.state_dict(), SAVE_DIR / "det_phase3.pth")
        print(f"  >> Best F1={best_val_f1:.4f}")
    else:
        patience += 1
        if patience >= CONFIG["patience"]: print(f"Early stopping"); break

print(f"Phase 3 done. Best F1: {best_val_f1:.4f}")

In [ ]:
# ===== PHASE 4: Joint Fine-tuning =====
print("\n" + "=" * 50)
print("PHASE 4: Joint Fine-tuning (AdaBelief, 0.1x LR)")
print("=" * 50)

for param in model.parameters(): param.requires_grad = True
optim_joint = make_adabelief(model.parameters(), lr=CONFIG["lr"] * 0.1)
best_joint_f1 = 0.0
patience = 0

for epoch in range(1, CONFIG["max_epochs_phase4"] + 1):
    model.train()
    all_y, all_p = [], []
    for caption, image, _ in tqdm(loaders["train"], desc=f"P4 Ep{epoch}"):
        caption, image = caption.to(DEVICE), image.to(DEVICE)
        bs = caption.size(0)
        if bs < 4: continue

        # Similarity
        cap_a, img_m, img_u = make_coolant_pairs(caption, image)
        ta_m, ia_m, _ = model.similarity_module(cap_a, img_m)
        ta_u, ia_u, _ = model.similarity_module(cap_a, img_u)
        ls = loss_cos(torch.cat([ta_m, ta_u]), torch.cat([ia_m, ia_u]),
            torch.cat([torch.ones(ta_m.size(0), device=DEVICE), -torch.ones(ta_u.size(0), device=DEVICE)]))

        # CLIP
        cap_pooled = caption.mean(dim=2)
        ie, te = model.clip_module(image, cap_pooled)
        logits = ie @ te.T * math.exp(0.07)
        ids = torch.arange(bs, device=DEVICE)
        l_clip = (loss_ce_clip(logits, ids) + loss_ce_clip(logits.T, ids)) / 2

        # Detection
        det_cap, det_img, det_labels = make_detection_batch(caption, image)
        det_cap_pooled = det_cap.mean(dim=2)
        ie2, te2 = model.clip_module(det_img, det_cap_pooled)
        det_out, attn, skl = model.detection_module(det_cap, det_img, te2, ie2)
        ld = loss_ce(det_out, det_labels) + 0.5 * loss_kl(F.log_softmax(attn, 1), F.softmax(skl, 1))

        total_loss = ls + l_clip + ld
        optim_joint.zero_grad(); total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        optim_joint.step()
        all_y.extend(det_labels.cpu().numpy()); all_p.extend(det_out.argmax(1).cpu().numpy())

    train_f1 = f1_score(all_y, all_p, average="macro", zero_division=0)

    # Validate
    model.eval()
    val_y, val_p = [], []
    with torch.no_grad():
        for caption, image, _ in loaders["dev"]:
            caption, image = caption.to(DEVICE), image.to(DEVICE)
            if caption.size(0) < 4: continue
            det_cap, det_img, det_labels = make_detection_batch(caption, image)
            cap_pooled = det_cap.mean(dim=2)
            ie, te = model.clip_module(det_img, cap_pooled)
            det_out, _, _ = model.detection_module(det_cap, det_img, te, ie)
            val_y.extend(det_labels.cpu().numpy()); val_p.extend(det_out.argmax(1).cpu().numpy())

    val_f1 = f1_score(val_y, val_p, average="macro", zero_division=0)
    prec, rec, _, _ = precision_recall_fscore_support(val_y, val_p, average=None, zero_division=0)
    cm = confusion_matrix(val_y, val_p)
    print(f"  Epoch {epoch}: train_F1={train_f1:.4f} val_F1={val_f1:.4f} CM={cm.tolist()}")

    csv_writer.writerow({"phase": 4, "epoch": epoch, "train_loss": "",
                         "train_f1": f"{train_f1:.4f}", "val_f1": f"{val_f1:.4f}",
                         "val_real_rec": f"{rec[0]:.3f}" if len(rec) > 0 else "",
                         "val_fake_rec": f"{rec[1]:.3f}" if len(rec) > 1 else ""})
    csv_file.flush()

    if val_f1 > best_joint_f1:
        best_joint_f1 = val_f1; patience = 0
        torch.save(model.state_dict(), SAVE_DIR / "full_model_phase4.pth")
        print(f"  >> Best F1={best_joint_f1:.4f}")
    else:
        patience += 1
        if patience >= CONFIG["patience"]: print(f"Early stopping"); break

csv_file.close()

print(f"\nAll phases complete.")
print(f"  Phase 1 CLIP loss: {best_clip_loss:.4f}")
print(f"  Phase 2 Sim loss:  {best_sim_loss:.4f}")
print(f"  Phase 3 Det F1:    {best_val_f1:.4f}")
print(f"  Phase 4 Joint F1:  {best_joint_f1:.4f}")
print(f"\nLog saved to {csv_path}")

In [ ]:
# Test evaluation on best joint model
checkpoint = torch.load(SAVE_DIR / "full_model_phase4.pth", map_location=DEVICE)
model.load_state_dict(checkpoint)
print("Loaded best Phase 4 model")

model.eval()
test_y, test_p = [], []
with torch.no_grad():
    for caption, image, _ in tqdm(loaders["test"], desc="Test"):
        caption, image = caption.to(DEVICE), image.to(DEVICE)
        if caption.size(0) < 4: continue
        det_cap, det_img, det_labels = make_detection_batch(caption, image)
        cap_pooled = det_cap.mean(dim=2)
        ie, te = model.clip_module(det_img, cap_pooled)
        det_out, _, _ = model.detection_module(det_cap, det_img, te, ie)
        test_y.extend(det_labels.cpu().numpy()); test_p.extend(det_out.argmax(1).cpu().numpy())

test_f1 = f1_score(test_y, test_p, average="macro", zero_division=0)
test_acc = accuracy_score(test_y, test_p)

print(f"\nTest: acc={test_acc:.4f} F1={test_f1:.4f}")
print(classification_report(test_y, test_p, target_names=["Real(matched)", "Fake(unmatched)"], digits=4))
print(f"Confusion Matrix: {confusion_matrix(test_y, test_p).tolist()}")

# Save test report
report_path = LOG_DIR / "test_report_phased.txt"
with open(report_path, "w") as f:
    f.write(f"Model: COOLANT + AdaBelief (Phased)\n")
    f.write(f"Phase 1 CLIP loss: {best_clip_loss:.4f}\n")
    f.write(f"Phase 2 Sim loss: {best_sim_loss:.4f}\n")
    f.write(f"Phase 3 Det F1: {best_val_f1:.4f}\n")
    f.write(f"Phase 4 Joint F1: {best_joint_f1:.4f}\n")
    f.write(f"Test F1: {test_f1:.4f}\n")
    f.write(f"Test Acc: {test_acc:.4f}\n\n")
    f.write(classification_report(test_y, test_p, target_names=["Real(matched)", "Fake(unmatched)"], digits=4))
    f.write(f"\nConfusion Matrix: {confusion_matrix(test_y, test_p).tolist()}\n")
    f.write(f"\nConfig:\n")
    for k, v in CONFIG.items():
        f.write(f"  {k}: {v}\n")

print(f"Report saved to {report_path}")